In [1]:
!pip install fugashi unidic-lite pandas scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [13]:
"""
phase1
人間が解釈できる特徴量のみで分類器を作る
->n-gram除去
"""
import glob
import os
import re
from collections import Counter
import numpy as np
import pandas as pd
from fugashi import Tagger
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# 形態素解析用の共通インスタンスと品詞定義
tagger = Tagger()
FUNCTION_POS = {"助詞", "助動詞", "接続詞", "連体詞"}
CONTENT_POS = {"名詞", "動詞", "形容詞", "形状詞", "副詞", "感動詞"}


def analyze_text_files_per_sentence(folder_path, label):
    """指定されたフォルダ内の全テキストファイルを読み込み、文単位で形態素解析と各種特徴量を抽出する"""
    # 数値順にソートしてファイルを取得
    search_path = os.path.join(folder_path, "*.txt")
    filepaths = sorted(
        glob.glob(search_path),
        key=lambda x: int(re.search(r"\d+", os.path.basename(x)).group()) if re.search(r"\d+", os.path.basename(x)) else 0,
    )

    tokenized_texts = []
    content_texts = []
    function_texts = []
    feature_data = []

    for filepath in filepaths:
        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()

        # まず文単位に分割（空の文字列を除外）
        sentences = re.split(r"[。！？]", text)
        sentences = [s.strip() for s in sentences if s.strip()]

        for s_idx, sentence_text in enumerate(sentences):
            pos_counter = Counter()
            particle_counter = Counter()
            tokens, content_words, function_words, all_words = [], [], [], []
            word_count = content_count = function_count = 0

            # 記号のカウント（1文あたり）
            comma_count = sentence_text.count("、")
            period_count = sentence_text.count("。")
            exclamation_count = sentence_text.count("！")
            question_count = sentence_text.count("？")

            # 文字長・文字種特徴
            total_chars = len(sentence_text)
            if total_chars == 0:
                continue  # 空の文はスキップ
                
            hiragana_count = len(re.findall(r"[ぁ-ん]", sentence_text))
            katakana_count = len(re.findall(r"[ァ-ン]", sentence_text))
            kanji_count = len(re.findall(r"[一-龥]", sentence_text))
            alnum_count = len(re.findall(r"[a-zA-Z0-9]", sentence_text))
            
            hiragana_ratio = hiragana_count / total_chars
            katakana_ratio = katakana_count / total_chars
            kanji_ratio = kanji_count / total_chars
            alnum_ratio = alnum_count / total_chars

            # 形態素解析（1文に対して実行）
            for word in tagger(sentence_text):
                pos = word.feature.pos1
                surface = word.surface

                if pos == "補助記号":
                    continue
                if pos == "助詞":
                    particle_counter[surface] += 1

                tokens.append(surface)
                all_words.append(surface)
                pos_counter[pos] += 1
                word_count += 1

                if pos in FUNCTION_POS:
                    function_words.append(surface)
                    function_count += 1
                elif pos in CONTENT_POS:
                    content_words.append(surface)
                    content_count += 1

            # 1文に単語が含まれない場合はスキップ
            if word_count == 0:
                continue

            # 基本割合・指標計算
            f_per = (function_count / word_count) * 100
            c_per = (content_count / word_count) * 100
            comma_ratio = comma_count / word_count
            period_ratio = period_count / word_count
            ttr = len(set(all_words)) / word_count

            # ===== Hapax / Yule =====
            hapax_all, yule_all = hapax_yule(all_words, word_count)
            hapax_func, yule_func = hapax_yule(function_words, function_count)
            hapax_cont, yule_cont = hapax_yule(content_words, content_count)

            # 助詞・品詞割合
            total_particle = sum(particle_counter.values())
            particle_ratio = {
                f"助詞_{k}": (v / total_particle if total_particle > 0 else 0)
                for k, v in particle_counter.items()
            }
            pos_ratio = {pos: (count / word_count) * 100 for pos, count in pos_counter.items()}

            # リストに追加（TF-IDFやN-gramを後続で計算する場合用）
            tokenized_texts.append(" ".join(tokens))
            function_texts.append(" ".join(function_words))
            content_texts.append(" ".join(content_words))

            # 特徴量データの辞書を作成（ファイル名に加えて文番号を付与）
            feature_data.append(
                {
                    "filename": f"{os.path.basename(filepath)}_s{s_idx}",
                    "label": label,
                    "文長": total_chars, # 文単位なので「総単語数」の代わりに「文長」
                    "総単語数": word_count,
                    "機能語率": f_per,
                    "内容語率": c_per,
                    "異なり語数": len(set(all_words)),
                    "TTR": ttr,
                    "ひらがな率": hiragana_ratio,
                    "カタカナ率": katakana_ratio,
                    "漢字率": kanji_ratio,
                    "英数字率": alnum_ratio,
                    "Hapax率_all": hapax_all,
                    "Yules_K_all": yule_all,
                    "Hapax率_func": hapax_func,
                    "Yules_K_func": yule_func,
                    "Hapax率_cont": hapax_cont,
                    "Yules_K_cont": yule_cont,
                    "名詞割合": pos_ratio.get("名詞", 0),
                    "動詞割合": pos_ratio.get("動詞", 0),
                    "形容詞割合": pos_ratio.get("形容詞", 0),
                    "形状詞割合": pos_ratio.get("形状詞", 0),
                    "助詞割合": pos_ratio.get("助詞", 0),
                    "助動詞割合": pos_ratio.get("助動詞", 0),
                    "副詞割合": pos_ratio.get("副詞", 0),
                    "連体詞割合": pos_ratio.get("連体詞", 0),
                    "接続詞割合": pos_ratio.get("接続詞", 0),
                    "感動詞割合": pos_ratio.get("感動詞", 0),
                    **particle_ratio,
                    "読点頻度": comma_ratio,
                    "句点頻度": period_ratio,
                    "感嘆符数": exclamation_count,
                    "疑問符数": question_count,
                }
            )

    return {
        "tokenized_texts": tokenized_texts,
        "content_texts": content_texts,
        "function_texts": function_texts,
        "feature_data": feature_data,
    }

def hapax_yule(tokens, word_count):
    if word_count == 0:
        return 0, 0

    freq = Counter(tokens)

    # Hapax率
    hapax_count = sum(1 for v in freq.values() if v == 1)
    hapax_ratio = hapax_count / word_count

    # Yule's K
    freq_counter = Counter(freq.values())
    m1 = word_count
    m2 = sum((f ** 2) * c for f, c in freq_counter.items())

    yules_k = 10000 * ((m2 - m1) / (m1 * m1)) if m1 > 0 else 0

    return hapax_ratio, yules_k

def jpn_word_ngram_analyzer(text):
    """半角スペースを挟ずに日本語の単語N-gram（2-gram, 3-gram）を生成する"""
    tokens = text.split()
    ngrams = []
    for n in [2, 3]:
        for i in range(len(tokens) - n + 1):
            ngrams.append("".join(tokens[i : i + n]))
    return ngrams


def print_analysis_results(
    data,
    X_content,
    content_features,
    Y_function,
    function_features,
    X_function_ngram,
    function_ngram_features,
    X_content_ngram,
    content_ngram_features,
    X_char_ngram,
    char_ngram_features,
    top_k=30,
):
    """解析結果を画面にわかりやすく表示する"""
    for i, (filepath, text) in enumerate(data["files"]):
        function_ngram_values = X_function_ngram.toarray()[i]
        content_ngram_values = X_content_ngram.toarray()[i]
        char_ngram_values = X_char_ngram.toarray()[i]

        # 各種N-gram上位の抽出
        function_ngram_top = dict(
            sorted(
                [
                    (f, int(v))
                    for f, v in zip(
                        function_ngram_features, function_ngram_values
                    )
                ],
                key=lambda x: x[1],
                reverse=True,
            )[:top_k]
        )
        content_ngram_top = dict(
            sorted(
                [
                    (f, int(v))
                    for f, v in zip(content_ngram_features, content_ngram_values)
                ],
                key=lambda x: x[1],
                reverse=True,
            )[:top_k]
        )
        char_top = dict(
            sorted(
                [
                    (f, int(v))
                    for f, v in zip(char_ngram_features, char_ngram_values)
                ],
                key=lambda x: x[1],
                reverse=True,
            )[:top_k]
        )

        data["feature_data"][i].update(
            {
                "function_ngram_top": function_ngram_top,
                "content_ngram_top": content_ngram_top,
                "char_ngram_top": char_top,
            }
        )

        print(f"\n=== ファイル：{filepath} ===")
        for key, value in data["feature_data"][i].items():
            if isinstance(value, float):
                print(f"{key}：{value:.4f}")
            else:
                print(f"{key}：{value}")

        print("\n=== 機能語 ===")
        print(data["function_texts"][i])
        print(f"カウント数：{data['function_counts'][i]}個")
        print(f"割合：{data['function_pers'][i]:.2f}%")
        for word, score in zip(function_features, Y_function.toarray()[i]):
            if score > 0:
                print(f"{word} {score:.4f}")

        print("\n=== 内容語 ===")
        print(data["content_texts"][i])
        print(f"カウント数：{data['content_counts'][i]}個")
        print(f"割合：{data['content_pers'][i]:.2f}%")
        for word, score in zip(content_features, X_content.toarray()[i]):
            if score > 0:
                print(f"{word} {score:.4f}")


if __name__ == "__main__":
    # フォルダパスを指定してテキスト解析を実行
    human_data = analyze_text_files_per_sentence(
        "train_Human_text",
        "Human",
    )
    
    ai_data = analyze_text_files_per_sentence(
        "train_AI_text",
        "AI"
    )

#     if not analysis_data["files"]:
#         print("指定されたフォルダにテキストファイルが見つかりませんでした。")
#     else:
#         # 特徴量抽出（ベクトライズ）の実行
#         content_vectorizer = TfidfVectorizer()
#         X_content = content_vectorizer.fit_transform(
#             analysis_data["content_texts"]
#         )
#         content_features = content_vectorizer.get_feature_names_out()

#         function_vectorizer = TfidfVectorizer()
#         Y_function = function_vectorizer.fit_transform(
#             analysis_data["function_texts"]
#         )
#         function_features = function_vectorizer.get_feature_names_out()

#         # 機能語の単語N-gram
#         function_ngram_vectorizer = CountVectorizer(
#             analyzer=jpn_word_ngram_analyzer, max_features=2000
#         )
#         X_function_ngram = function_ngram_vectorizer.fit_transform(
#             analysis_data["function_texts"]
#         )
#         function_ngram_features = (
#             function_ngram_vectorizer.get_feature_names_out()
#         )

#         # 内容語の単語N-gram
#         content_ngram_vectorizer = CountVectorizer(
#             analyzer=jpn_word_ngram_analyzer, max_features=2000
#         )
#         X_content_ngram = content_ngram_vectorizer.fit_transform(
#             analysis_data["content_texts"]
#         )
#         content_ngram_features = (
#             content_ngram_vectorizer.get_feature_names_out()
#         )

#         # 文字N-gram (元の生テキストを使用)
#         raw_texts = [text for _, text in analysis_data["files"]]
#         char_ngram_vectorizer = CountVectorizer(
#             analyzer="char", ngram_range=(3, 5), max_features=3000
#         )
#         X_char_ngram = char_ngram_vectorizer.fit_transform(raw_texts)
#         char_ngram_features = char_ngram_vectorizer.get_feature_names_out()

#         # 結果の表示
#         print_analysis_results(
#             analysis_data,
#             X_content,
#             content_features,
#             Y_function,
#             function_features,
#             X_function_ngram,
#             function_ngram_features,
#             X_content_ngram,
#             content_ngram_features,
#             X_char_ngram,
#             char_ngram_features,
#         )
        
    all_features = (
        human_data["feature_data"]
        + ai_data["feature_data"]
    )

    df = pd.DataFrame(all_features)

    # df = df.sample(
    #     frac=1,
    #     random_state=42
    # ).reset_index(drop=True)

    df.to_csv(
        "train_Confidence-Level_one_sentence_stylometry_features.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(df.head())
    print(f"総データ数：{len(df)}")

               filename  label  文長  総単語数       機能語率       内容語率  異なり語数  \
0  Human_text_01.txt_s0  Human  22    12  33.333333  66.666667     12   
1  Human_text_01.txt_s1  Human  49    24  45.833333  54.166667     18   
2  Human_text_01.txt_s2  Human  49    24  41.666667  58.333333     23   
3  Human_text_01.txt_s3  Human  73    41  51.219512  46.341463     30   
4  Human_text_01.txt_s4  Human  29    14  50.000000  50.000000     14   

        TTR     ひらがな率     カタカナ率  ...  助詞_のみ  助詞_しも  助詞_しか  助詞_すら  助詞_より  \
0  1.000000  0.363636  0.272727  ...    NaN    NaN    NaN    NaN    NaN   
1  0.750000  0.326531  0.387755  ...    NaN    NaN    NaN    NaN    NaN   
2  0.958333  0.326531  0.285714  ...    NaN    NaN    NaN    NaN    NaN   
3  0.731707  0.452055  0.273973  ...    NaN    NaN    NaN    NaN    NaN   
4  1.000000  0.275862  0.310345  ...    NaN    NaN    NaN    NaN    NaN   

   助詞_こそ  助詞_つつ  助詞_くらい  助詞_かしら  助詞_ずつ  
0    NaN    NaN     NaN     NaN    NaN  
1    NaN    NaN     NaN     

In [15]:
"""
phase1
人間が解釈できる特徴量のみで分類器を作る
->n-gram除去
"""
import glob
import os
import re
from collections import Counter
import numpy as np
import pandas as pd
from fugashi import Tagger
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# 形態素解析用の共通インスタンスと品詞定義
tagger = Tagger()
FUNCTION_POS = {"助詞", "助動詞", "接続詞", "連体詞"}
CONTENT_POS = {"名詞", "動詞", "形容詞", "形状詞", "副詞", "感動詞"}


def analyze_text_files_per_sentence(folder_path, label):
    """指定されたフォルダ内の全テキストファイルを読み込み、文単位で形態素解析と各種特徴量を抽出する"""
    # 数値順にソートしてファイルを取得
    search_path = os.path.join(folder_path, "*.txt")
    filepaths = sorted(
        glob.glob(search_path),
        key=lambda x: int(re.search(r"\d+", os.path.basename(x)).group()) if re.search(r"\d+", os.path.basename(x)) else 0,
    )

    tokenized_texts = []
    content_texts = []
    function_texts = []
    feature_data = []

    for filepath in filepaths:
        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()

        # まず文単位に分割（空の文字列を除外）
        sentences = re.split(r"[。！？]", text)
        sentences = [s.strip() for s in sentences if s.strip()]

        for s_idx in range(len(sentences)):
            sentence_text = "。".join(sentences[:s_idx+1]) + "。"
            pos_counter = Counter()
            particle_counter = Counter()
            tokens, content_words, function_words, all_words = [], [], [], []
            word_count = content_count = function_count = 0

            # 記号のカウント（1文あたり）
            comma_count = sentence_text.count("、")
            period_count = sentence_text.count("。")
            exclamation_count = sentence_text.count("！")
            question_count = sentence_text.count("？")

            # 文字長・文字種特徴
            total_chars = len(sentence_text)
            if total_chars == 0:
                continue  # 空の文はスキップ
                
            hiragana_count = len(re.findall(r"[ぁ-ん]", sentence_text))
            katakana_count = len(re.findall(r"[ァ-ン]", sentence_text))
            kanji_count = len(re.findall(r"[一-龥]", sentence_text))
            alnum_count = len(re.findall(r"[a-zA-Z0-9]", sentence_text))
            
            hiragana_ratio = hiragana_count / total_chars
            katakana_ratio = katakana_count / total_chars
            kanji_ratio = kanji_count / total_chars
            alnum_ratio = alnum_count / total_chars

            # 形態素解析（1文に対して実行）
            for word in tagger(sentence_text):
                pos = word.feature.pos1
                surface = word.surface

                if pos == "補助記号":
                    continue
                if pos == "助詞":
                    particle_counter[surface] += 1

                tokens.append(surface)
                all_words.append(surface)
                pos_counter[pos] += 1
                word_count += 1

                if pos in FUNCTION_POS:
                    function_words.append(surface)
                    function_count += 1
                elif pos in CONTENT_POS:
                    content_words.append(surface)
                    content_count += 1

            # 1文に単語が含まれない場合はスキップ
            if word_count == 0:
                continue

            # 基本割合・指標計算
            f_per = (function_count / word_count) * 100
            c_per = (content_count / word_count) * 100
            comma_ratio = comma_count / word_count
            period_ratio = period_count / word_count
            ttr = len(set(all_words)) / word_count

            # ===== Hapax / Yule =====
            hapax_all, yule_all = hapax_yule(all_words, word_count)
            hapax_func, yule_func = hapax_yule(function_words, function_count)
            hapax_cont, yule_cont = hapax_yule(content_words, content_count)

            # 助詞・品詞割合
            total_particle = sum(particle_counter.values())
            particle_ratio = {
                f"助詞_{k}": (v / total_particle if total_particle > 0 else 0)
                for k, v in particle_counter.items()
            }
            pos_ratio = {pos: (count / word_count) * 100 for pos, count in pos_counter.items()}

            # リストに追加（TF-IDFやN-gramを後続で計算する場合用）
            tokenized_texts.append(" ".join(tokens))
            function_texts.append(" ".join(function_words))
            content_texts.append(" ".join(content_words))

            # 特徴量データの辞書を作成（ファイル名に加えて文番号を付与）
            feature_data.append(
                {
                    "filename": f"{os.path.basename(filepath)}_s{s_idx}",
                    "label": label,
                    "文長": total_chars, # 文単位なので「総単語数」の代わりに「文長」
                    "総単語数": word_count,
                    "機能語率": f_per,
                    "内容語率": c_per,
                    "異なり語数": len(set(all_words)),
                    "TTR": ttr,
                    "ひらがな率": hiragana_ratio,
                    "カタカナ率": katakana_ratio,
                    "漢字率": kanji_ratio,
                    "英数字率": alnum_ratio,
                    "Hapax率_all": hapax_all,
                    "Yules_K_all": yule_all,
                    "Hapax率_func": hapax_func,
                    "Yules_K_func": yule_func,
                    "Hapax率_cont": hapax_cont,
                    "Yules_K_cont": yule_cont,
                    "名詞割合": pos_ratio.get("名詞", 0),
                    "動詞割合": pos_ratio.get("動詞", 0),
                    "形容詞割合": pos_ratio.get("形容詞", 0),
                    "形状詞割合": pos_ratio.get("形状詞", 0),
                    "助詞割合": pos_ratio.get("助詞", 0),
                    "助動詞割合": pos_ratio.get("助動詞", 0),
                    "副詞割合": pos_ratio.get("副詞", 0),
                    "連体詞割合": pos_ratio.get("連体詞", 0),
                    "接続詞割合": pos_ratio.get("接続詞", 0),
                    "感動詞割合": pos_ratio.get("感動詞", 0),
                    **particle_ratio,
                    "読点頻度": comma_ratio,
                    "句点頻度": period_ratio,
                    "感嘆符数": exclamation_count,
                    "疑問符数": question_count,
                }
            )

    return {
        "tokenized_texts": tokenized_texts,
        "content_texts": content_texts,
        "function_texts": function_texts,
        "feature_data": feature_data,
    }

def hapax_yule(tokens, word_count):
    if word_count == 0:
        return 0, 0

    freq = Counter(tokens)

    # Hapax率
    hapax_count = sum(1 for v in freq.values() if v == 1)
    hapax_ratio = hapax_count / word_count

    # Yule's K
    freq_counter = Counter(freq.values())
    m1 = word_count
    m2 = sum((f ** 2) * c for f, c in freq_counter.items())

    yules_k = 10000 * ((m2 - m1) / (m1 * m1)) if m1 > 0 else 0

    return hapax_ratio, yules_k

def jpn_word_ngram_analyzer(text):
    """半角スペースを挟ずに日本語の単語N-gram（2-gram, 3-gram）を生成する"""
    tokens = text.split()
    ngrams = []
    for n in [2, 3]:
        for i in range(len(tokens) - n + 1):
            ngrams.append("".join(tokens[i : i + n]))
    return ngrams


def print_analysis_results(
    data,
    X_content,
    content_features,
    Y_function,
    function_features,
    X_function_ngram,
    function_ngram_features,
    X_content_ngram,
    content_ngram_features,
    X_char_ngram,
    char_ngram_features,
    top_k=30,
):
    """解析結果を画面にわかりやすく表示する"""
    for i, (filepath, text) in enumerate(data["files"]):
        function_ngram_values = X_function_ngram.toarray()[i]
        content_ngram_values = X_content_ngram.toarray()[i]
        char_ngram_values = X_char_ngram.toarray()[i]

        # 各種N-gram上位の抽出
        function_ngram_top = dict(
            sorted(
                [
                    (f, int(v))
                    for f, v in zip(
                        function_ngram_features, function_ngram_values
                    )
                ],
                key=lambda x: x[1],
                reverse=True,
            )[:top_k]
        )
        content_ngram_top = dict(
            sorted(
                [
                    (f, int(v))
                    for f, v in zip(content_ngram_features, content_ngram_values)
                ],
                key=lambda x: x[1],
                reverse=True,
            )[:top_k]
        )
        char_top = dict(
            sorted(
                [
                    (f, int(v))
                    for f, v in zip(char_ngram_features, char_ngram_values)
                ],
                key=lambda x: x[1],
                reverse=True,
            )[:top_k]
        )

        data["feature_data"][i].update(
            {
                "function_ngram_top": function_ngram_top,
                "content_ngram_top": content_ngram_top,
                "char_ngram_top": char_top,
            }
        )

        print(f"\n=== ファイル：{filepath} ===")
        for key, value in data["feature_data"][i].items():
            if isinstance(value, float):
                print(f"{key}：{value:.4f}")
            else:
                print(f"{key}：{value}")

        print("\n=== 機能語 ===")
        print(data["function_texts"][i])
        print(f"カウント数：{data['function_counts'][i]}個")
        print(f"割合：{data['function_pers'][i]:.2f}%")
        for word, score in zip(function_features, Y_function.toarray()[i]):
            if score > 0:
                print(f"{word} {score:.4f}")

        print("\n=== 内容語 ===")
        print(data["content_texts"][i])
        print(f"カウント数：{data['content_counts'][i]}個")
        print(f"割合：{data['content_pers'][i]:.2f}%")
        for word, score in zip(content_features, X_content.toarray()[i]):
            if score > 0:
                print(f"{word} {score:.4f}")


if __name__ == "__main__":
    # フォルダパスを指定してテキスト解析を実行
    human_data = analyze_text_files_per_sentence(
        "train_Human_text",
        "Human",
    )
    
    ai_data = analyze_text_files_per_sentence(
        "train_AI_text",
        "AI"
    )

#     if not analysis_data["files"]:
#         print("指定されたフォルダにテキストファイルが見つかりませんでした。")
#     else:
#         # 特徴量抽出（ベクトライズ）の実行
#         content_vectorizer = TfidfVectorizer()
#         X_content = content_vectorizer.fit_transform(
#             analysis_data["content_texts"]
#         )
#         content_features = content_vectorizer.get_feature_names_out()

#         function_vectorizer = TfidfVectorizer()
#         Y_function = function_vectorizer.fit_transform(
#             analysis_data["function_texts"]
#         )
#         function_features = function_vectorizer.get_feature_names_out()

#         # 機能語の単語N-gram
#         function_ngram_vectorizer = CountVectorizer(
#             analyzer=jpn_word_ngram_analyzer, max_features=2000
#         )
#         X_function_ngram = function_ngram_vectorizer.fit_transform(
#             analysis_data["function_texts"]
#         )
#         function_ngram_features = (
#             function_ngram_vectorizer.get_feature_names_out()
#         )

#         # 内容語の単語N-gram
#         content_ngram_vectorizer = CountVectorizer(
#             analyzer=jpn_word_ngram_analyzer, max_features=2000
#         )
#         X_content_ngram = content_ngram_vectorizer.fit_transform(
#             analysis_data["content_texts"]
#         )
#         content_ngram_features = (
#             content_ngram_vectorizer.get_feature_names_out()
#         )

#         # 文字N-gram (元の生テキストを使用)
#         raw_texts = [text for _, text in analysis_data["files"]]
#         char_ngram_vectorizer = CountVectorizer(
#             analyzer="char", ngram_range=(3, 5), max_features=3000
#         )
#         X_char_ngram = char_ngram_vectorizer.fit_transform(raw_texts)
#         char_ngram_features = char_ngram_vectorizer.get_feature_names_out()

#         # 結果の表示
#         print_analysis_results(
#             analysis_data,
#             X_content,
#             content_features,
#             Y_function,
#             function_features,
#             X_function_ngram,
#             function_ngram_features,
#             X_content_ngram,
#             content_ngram_features,
#             X_char_ngram,
#             char_ngram_features,
#         )
        
    all_features = (
        human_data["feature_data"]
        + ai_data["feature_data"]
    )

    df = pd.DataFrame(all_features)

    # df = df.sample(
    #     frac=1,
    #     random_state=42
    # ).reset_index(drop=True)

    df.to_csv(
        "train_Confidence-Level_one_sentence_stylometry_features.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(df.head())
    print(f"総データ数：{len(df)}")

               filename  label   文長  総単語数       機能語率       内容語率  異なり語数  \
0  Human_text_01.txt_s0  Human   23    12  33.333333  66.666667     12   
1  Human_text_01.txt_s1  Human   73    36  41.666667  58.333333     25   
2  Human_text_01.txt_s2  Human  123    60  41.666667  58.333333     38   
3  Human_text_01.txt_s3  Human  197   101  45.544554  53.465347     54   
4  Human_text_01.txt_s4  Human  227   115  46.086957  53.043478     62   

        TTR     ひらがな率     カタカナ率  ...  助詞_のみ  助詞_しも  助詞_しか  助詞_すら  助詞_より  \
0  1.000000  0.347826  0.260870  ...    NaN    NaN    NaN    NaN    NaN   
1  0.694444  0.328767  0.342466  ...    NaN    NaN    NaN    NaN    NaN   
2  0.633333  0.325203  0.317073  ...    NaN    NaN    NaN    NaN    NaN   
3  0.534653  0.370558  0.299492  ...    NaN    NaN    NaN    NaN    NaN   
4  0.539130  0.356828  0.299559  ...    NaN    NaN    NaN    NaN    NaN   

   助詞_こそ  助詞_つつ  助詞_くらい  助詞_かしら  助詞_ずつ  
0    NaN    NaN     NaN     NaN    NaN  
1    NaN    NaN     Na